#### Name: Blessing Adeniji
#### Degree: MSc Artifical Intelligence Online
#### Capstone Project: AI-Generated Text Detection - Deepfakes

Step 4: DeBERTa-v3-base spot-check (using LLMs): Fine-tune on MAGE only. 
Purpose: To confirm the capacity-independence finding at 184 params
In the end RoBERTa was used because DeBERTa failed and could not be trained stably in the environment. So RoBERTa was a replacement.

In [4]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score


In [7]:
# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.version.cuda)

CUDA available: True
12.8


In [5]:
# Fine-tune DeBERT-v3 on MAGE only
# Load the MAGE train/val splits
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

# Load the DeBERTa tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

# Convert to HuggingFace Format and tokenize
mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_val_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

# Tiny-sanity check: can DeBERTa learn ANYTHING on 1000 samples?
tiny_train = Dataset.from_pandas(mage_train_dataset.head(1000)).map(tokenize, batched=True)
tiny_val = Dataset.from_pandas(mage_validation_dataset.head(500)).map(tokenize, batched=True)

# Load the fresh base model
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)

# Training settings for MAGE
training_args = TrainingArguments(
    output_dir="models/deberta_base__mage",  # output directory - where checkpoints and model will be saved
    num_train_epochs=2,                 # number of training epochs - 2 for spot check
    per_device_train_batch_size=8,     # batch size for training - same as ModernBERT setting
    per_device_eval_batch_size=16,      # batch size for evaluation - same as ModernBERT setting
    learning_rate=5e-6,                 # learning rate - changed it to halved, deberta-v3 is LR-sensitive
    eval_strategy="epoch",              # evaluate each epoch
    save_strategy="epoch",              # save each epoch          
    load_best_model_at_end=True,        # load the best model when finished training (default metric is loss)
    bf16=True,                          
    warmup_steps=2000,                  # ~6% of total steps - to prevent collapse early
    logging_steps=50,
    report_to="none",
)

# Train the MAGE dataset
trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=mage_train_ds,         # training dataset
    eval_dataset=mage_val_ds,            # evaluation dataset
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),         # function to collate data into batches
    compute_metrics=compute_metrics      # function to compute metrics for evaluation
)

# Train the MAGE-trained model
trainer.train()

Map:   0%|          | 0/130645 [00:00<?, ? examples/s]

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.693363,0.693728,0.499982,0.666651
2,0.693673,0.693154,0.499982,0.666651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=32662, training_loss=0.6945784195457376, metrics={'train_runtime': 7989.6939, 'train_samples_per_second': 32.703, 'train_steps_per_second': 4.088, 'total_flos': 6.196033963555266e+16, 'train_loss': 0.6945784195457376, 'epoch': 2.0})

In [ ]:
import pandas as pd, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    p, l = eval_pred
    pr = np.argmax(p, axis=1)
    return {"accuracy": accuracy_score(l, pr), "f1": f1_score(l, pr)}

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
def tokenize(b): return tokenizer(b["text"], truncation=True, max_length=512)

tiny_train = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_train.csv").head(1000)).map(tokenize, batched=True)
tiny_val = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_val.csv").head(500)).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-base", num_labels=2)

training_args = TrainingArguments(
    output_dir="tmp_tiny", 
    num_train_epochs=3, 
    per_device_train_batch_size=8,
    learning_rate=2e-5, 
    eval_strategy="epoch", 
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=tiny_train, 
    eval_dataset=tiny_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics=compute_metrics)
trainer.train()

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.000000,nan,0.492000,0.000000
2,0.000000,nan,0.492000,0.000000
3,0.000000,nan,0.492000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.11446972147623698, metrics={'train_runtime': 82.666, 'train_samples_per_second': 36.291, 'train_steps_per_second': 4.536, 'total_flos': 716493030759072.0, 'train_loss': 0.11446972147623698, 'epoch': 3.0})

In [9]:
# RoBERTa-base spot-check on MAGE (replacing DeBERTa-v3, which produced NaN outputs
# in this transformers version — see methods note)
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_val_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

training_args = TrainingArguments(
    output_dir="models/roberta_base_mage",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=mage_train_ds, 
    eval_dataset=mage_val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics=compute_metrics
)

trainer.train()

Map:   0%|          | 0/130645 [00:00<?, ? examples/s]

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.302642,0.309883,0.926701,0.930233
2,0.176419,0.572347,0.900125,0.908020
3,0.077002,0.637610,0.908841,0.915407


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=48993, training_loss=0.16695133302714119, metrics={'train_runtime': 5678.1962, 'train_samples_per_second': 69.025, 'train_steps_per_second': 8.628, 'total_flos': 9.301398840219696e+16, 'train_loss': 0.16695133302714119, 'epoch': 3.0})

In [2]:
# Build a 4x4 Matrix
def save_results_to_csv(model_name, trained_on, tested_on, results):
    # Create a one-row table with the results
    row = pd.DataFrame({
        'model_name': [model_name],
        'trained_on': [trained_on],
        'tested_on': [tested_on],
        'accuracy': [results['eval_accuracy']],
        'f1': [results['eval_f1']],
        'loss': [results['eval_loss']]
    })

    # Append to results and create if it doesn't exist
    file_exists = os.path.isfile('all_models_evaluation_results.csv')
    row.to_csv('all_models_evaluation_results.csv', mode='a', header=not file_exists, index=False)
    print("Saved:", model_name, trained_on, tested_on)

In [10]:
# Save the DeBERTa-v3 fine-tuned MAGE model and tokenizer
model.save_pretrained("models/roberta_mage_final")
tokenizer.save_pretrained("models/roberta_mage_final")

# Load and tokenize all 4 test sets with the DeBERTa-v3 tokenizer
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)

# Evaluate on all 4 and save
save_results_to_csv("roberta-base", "MAGE", "Mage", trainer.evaluate(mage_test_ds))
save_results_to_csv("roberta-base", "MAGE", "ChatGPT-Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("roberta-base", "MAGE", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("roberta-base", "MAGE", "RAID", trainer.evaluate(raid_test_ds))


print("\nRoBERTa spot-check complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

...

Saved: roberta-base MAGE ChatGPT-Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.077002,0.910137,3,0.795400,0.815731


Saved: roberta-base MAGE Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.077002,1.480221,3,0.724586,0.762814


Saved: roberta-base MAGE RAID

RoBERTa spot-check complete
